# Product Recommendation System

This notebook builds a **content-based recommendation engine** for an e-commerce product catalog (Flipkart product dump). The idea is simple: instead of relying on user ratings or purchase history (which this dataset doesn't really have — most ratings are missing), we recommend products that are *textually similar* to a given product, using its name, brand, category and description.

Rough plan for the notebook:
1. Load and clean the raw product data
2. Engineer a combined text field that captures what a product "is"
3. Convert that text into TF-IDF vectors
4. Compute cosine similarity between products
5. Write a function that returns the top-N most similar products for a given item


In [ ]:
pip install nltk

First, the usual imports. `nltk` gets installed in case any text-cleaning steps need it later (stopwords/tokenizers), and `numpy`/`pandas` handle the actual data wrangling.


In [2]:
import numpy as np
import pandas as pd
import os

# Data Preprocessing

## Loading the data

Reading in the raw product catalog. This is a Flipkart-style dataset with ~20,000 product listings, each with fields like name, category tree, price, brand, description, and ratings.


In [3]:
products = pd.read_csv('products.csv')
products.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


A quick sanity check — how many *unique* product names vs. unique IDs are there? If `product_name` has far fewer uniques than `uniq_id`, that tells us the catalog has a lot of duplicate/near-duplicate listings (same product listed multiple times, e.g. different sellers or variants), which we'll need to handle before building the recommender.


In [4]:
len(products['product_name'].unique()), len(products['uniq_id'].unique())

(12676, 20000)

Checking the overall structure of the dataframe — column types and how much data is missing. A few columns like `retail_price`, `brand`, and `product_specifications` have noticeable null counts, which matters because we plan to concatenate several text columns together later — a stray `NaN` would break that.


In [5]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   uniq_id                  20000 non-null  object 
 1   crawl_timestamp          20000 non-null  object 
 2   product_url              20000 non-null  object 
 3   product_name             20000 non-null  object 
 4   product_category_tree    20000 non-null  object 
 5   pid                      20000 non-null  object 
 6   retail_price             19922 non-null  float64
 7   discounted_price         19922 non-null  float64
 8   image                    19997 non-null  object 
 9   is_FK_Advantage_product  20000 non-null  bool   
 10  description              19998 non-null  object 
 11  product_rating           20000 non-null  object 
 12  overall_rating           20000 non-null  object 
 13  brand                    14136 non-null  object 
 14  product_specifications

## Cleaning up `product_category_tree`

The category column comes in as a single messy string like `["Clothing >> Women's Clothing >> Lingerie..."]`, so this strips the surrounding brackets/quotes and splits it on `>>` into a proper list of category levels (e.g. `Clothing`, `Women's Clothing`, `Lingerie`, ...).


In [6]:
products['product_category_tree']=products['product_category_tree'].map(lambda x:x.strip('[]'))
products['product_category_tree']=products['product_category_tree'].map(lambda x:x.strip('"'))
products['product_category_tree']=products['product_category_tree'].map(lambda x:x.split('>>'))

## Dropping columns we don't need

For a *content-based* recommender, pricing, images, crawl timestamps and ratings don't add anything — the similarity is going to be based purely on text (name, category, brand, description). Dropping those columns keeps things lighter and avoids clutter.


In [7]:
products = products.drop(['crawl_timestamp','product_url','image',"retail_price","discounted_price","is_FK_Advantage_product","product_rating","overall_rating","product_specifications"], axis=1)

The dataframe now only has the columns that actually matter for computing text similarity.


In [8]:
products.head()

,uniq_id,product_name,product_category_tree,pid,description,brand
0,c2d766ca982eca8304150849735ffef9,Alisha Solid Women's Cycling Shorts,"[Clothing , Women's Clothing , Lingerie, Sle...",SRTEH2FF9KEDEFGF,Key Features of Alisha Solid Women's Cycling S...,Alisha
1,7f7036a6d550aaa89d34c77bd39a5e48,FabHomeDecor Fabric Double Sofa Bed,"[Furniture , Living Room Furniture , Sofa Be...",SBEEH3QGU7MFYJFY,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,FabHomeDecor
2,f449ec65dcbc041b6ae5e6a32717d01b,AW Bellies,"[Footwear , Women's Footwear , Ballerinas , ...",SHOEH4GRSUBJGZXE,Key Features of AW Bellies Sandals Wedges Heel...,AW
3,0973b37acd0c664e3de26e97e5571454,Alisha Solid Women's Cycling Shorts,"[Clothing , Women's Clothing , Lingerie, Sle...",SRTEH2F6HUZMQ6SJ,Key Features of Alisha Solid Women's Cycling S...,Alisha
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,Sicons All Purpose Arnica Dog Shampoo,"[Pet Supplies , Grooming , Skin & Coat Care ...",PSOEH3ZYDMSYARJ5,Specifications of Sicons All Purpose Arnica Do...,Sicons


## Handling duplicate products

Since several rows share the exact same `product_name` (as we saw earlier), a separate deduplicated copy `smd` is created here — keeping only the first occurrence of each product name. This will be the dataframe used for the combined-metadata similarity matrix later, so we're not comparing a product against near-identical copies of itself.


In [9]:
smd=products.copy()
# drop duplicate produts
smd.drop_duplicates(subset ="product_name", 
                     keep = "first", inplace = True)
smd.shape

(12676, 6)

The category tree is currently a list per row — flattening it back into a single string here so it can be concatenated with the other text fields.


In [10]:
products['product_category_tree'] = products['product_category_tree'].apply(lambda x: ''.join(x))

Just eyeballing the flattened category strings to confirm they look right before moving on.


In [11]:
products['product_category_tree']

0        Clothing  Women's Clothing  Lingerie, Sleep & ...
1        Furniture  Living Room Furniture  Sofa Beds & ...
2        Footwear  Women's Footwear  Ballerinas  AW Bel...
3        Clothing  Women's Clothing  Lingerie, Sleep & ...
4        Pet Supplies  Grooming  Skin & Coat Care  Sham...
                               ...                        
19995    Baby Care  Baby & Kids Gifts  Stickers  WallDe...
19996    Baby Care  Baby & Kids Gifts  Stickers  Wallma...
19997    Baby Care  Baby & Kids Gifts  Stickers  Elite ...
19998    Baby Care  Baby & Kids Gifts  Stickers  Elite ...
19999    Baby Care  Baby & Kids Gifts  Stickers  Elite ...
Name: product_category_tree, Length: 20000, dtype: object

#### Creating a combined column containing brand, description product name and product category

In [12]:
smd["all_meta"] = smd['product_name']+' '+ smd['brand']+ ' '+ products['product_category_tree']+smd['description']
smd["all_meta"] = smd["all_meta"].fillna('')
smd["all_meta"] = smd["all_meta"].apply(lambda x: x.lower())

Peeking at a few rows of `all_meta` to confirm the concatenation worked and everything's lowercased consistently.


In [13]:
smd["all_meta"].head()

0    alisha solid women's cycling shorts alisha clo...
1    fabhomedecor fabric double sofa bed fabhomedec...
2    aw bellies aw footwear  women's footwear  ball...
4    sicons all purpose arnica dog shampoo sicons p...
5    eternal gandhi super series crystal paper weig...
Name: all_meta, dtype: object

## Vectorizing the text (TF-IDF)

Setting up a `TfidfVectorizer` to turn the raw text into numeric vectors. A few choices worth noting:
- `ngram_range=(1,3)` — captures single words as well as short phrases (unigrams to trigrams), so "cycling shorts" is treated as more than just two unrelated words.
- `min_df=10` — ignores terms that appear in fewer than 10 products, which cuts down noise from rare/typo-ish tokens.
- `stop_words='english'` — removes common filler words ("the", "is", "in", etc.) that don't help distinguish products.

Also filling any missing `description` values with an empty string so `fit_transform` doesn't choke on `NaN`.


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
tfv = TfidfVectorizer(max_features=None,
                     strip_accents='unicode',
                     analyzer='word',
                     min_df=10,
                     token_pattern=r'\w{1,}',
                     ngram_range=(1,3),#take the combination of 1-3 different kind of words
                     stop_words='english')#removes all the unnecessary characters like the,in etc.
products['description'] = products['description'].fillna('')

#### Creating two matrices, one using just the product description and the other using description, brand and product category

## Building two different similarity matrices

Two TF-IDF matrices are fit here, on purpose:
- `tfv_matrix` — built from **description only**
- `tfidf_matrix` — built from the **combined metadata** (`all_meta`: name + brand + category + description)

This lets us compare recommendations based purely on how a product is *described* vs. recommendations that also account for its brand and category — useful for seeing which signal gives more sensible results.


In [15]:
tfv_matrix = tfv.fit_transform(products['description'])
tfidf_matrix = tfv.fit_transform(smd['all_meta'])

This is what the TF-IDF output looks like — a sparse matrix, since most products only use a small fraction of the overall vocabulary. `20000 x 23315` means 20,000 products mapped into a vocabulary of ~23,315 terms.


In [16]:
tfv_matrix

<20000x23315 sparse matrix of type '<class 'numpy.float64'>'
	with 1510389 stored elements in Compressed Sparse Row format>

Converting to a dense array just to visualize the actual numbers (mostly zeros, as expected for TF-IDF on a large vocabulary — not something you'd want to do on the full matrix in a memory-constrained setting, but fine here for inspection).


In [17]:
tfv_matrix.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

Confirming the shape: 20,000 products, ~23,315 unique terms/n-grams in the vocabulary.


In [18]:
tfv_matrix.shape

(20000, 23315)

# Cosine Similarity Algorithm 

Using `cosine_similarity` to compare every product against every other product in vector space. This gives two similarity matrices — `des_sim` (description-only) and `overall_sim` (combined metadata) — each of shape (n_products x n_products), where a value closer to 1 means two products are more textually similar.


In [19]:
from sklearn.metrics.pairwise import cosine_similarity
des_sim = cosine_similarity(tfv_matrix, tfv_matrix)
overall_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

Just a quick exploratory check — `enumerate` pairs each product's index with its similarity score against product #1, which is the basic building block the recommendation function below will use (sort these pairs by score to find the closest matches).


In [21]:
enumerate(overall_sim[1])

## Mapping product names to their row index

Since the similarity matrices are indexed by row number (not product name), this builds a lookup — `product_name -> row index` — so that later we can look up a product by name (or index) and find its position in the similarity matrix. `drop_duplicates()` guards against the same product name mapping to multiple indices.


In [22]:
indices = pd.Series(products.index,index=products['product_name']).drop_duplicates()

Sanity check — confirming what product actually sits at index 20, since that's the one used to test the recommender below.


In [23]:
products['product_name'].iloc[20]

'Sicons Conditioning Conditoner Dog Shampoo'

A couple of extra imports (`pairwise_distances`, `MultiLabelBinarizer`) — kept here as they're useful if the similarity approach is extended later (e.g. distance metrics other than cosine, or one-hot encoding categorical features like category tags).


In [24]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import MultiLabelBinarizer

# Recommending Products

## The recommendation function

This is the core piece: given a product's index (`title`, despite the name, is used as an index here) and a similarity matrix, it:
1. Looks up that product's similarity scores against every other product
2. Sorts all products by similarity score, descending
3. Skips the first result (a product is always most similar to itself) and takes the next top 10
4. Returns the names of those top 10 products


In [26]:
def product_recommendation(title, matrix):
    idx = indices[title]
    sig_scores = list(enumerate(matrix[idx]))
    sig_scores = sorted(sig_scores, key = lambda x: x[1], reverse=True )
    sig_scores = sig_scores[1:11]
    product_indices = [i[0] for i in sig_scores]
    return products['product_name'].iloc[product_indices]

## Testing it out

Trying the recommender on product #20 (`Sicons Conditioning Conditoner Dog Shampoo`) using `des_sim` (description-only similarity). The results are almost entirely other pet-care products — a good sign that description-based similarity alone captures the right neighborhood, even before factoring in brand/category.


In [27]:
product_recommendation(20, des_sim).unique()

array(['Sicons All Purpose Arnica Dog Shampoo',
       'Sicons All Purpose Tea Tree Dog Shampoo',
       'Magideal Raincoat for Dog',
       'Royal Canin Maxi Starter 1kg Vegetable Dog Food',
       'Four Paws Sweats for Cat, Dog', 'Babyoye Premium Dog Face Bib',
       'Petto Raincoat for Dog, Cat',
       'Wella Elements Leight Weight Renewing Conditioner',
       'L Oreal Eversleek Sulfate - Free Smoothing System Intense Smoothing Shampoo',
       'Four Paws Round Plastic Pet Bottle'], dtype=object)

Now testing product #2 (`AW Bellies`) using `overall_sim` (combined name + brand + category + description). The results here are far less coherent — jewelry, car covers, phone cases — which suggests that blending in raw category/brand text without more careful weighting can actually dilute the similarity signal rather than improve it. Worth digging into further (e.g. weighting fields differently, or checking whether the category-tree cleanup upstream introduced noise).


In [28]:
product_recommendation(2, overall_sim).head(5)

11780    Radhesh Creation Crystal Yellow Gold Plated Br...
10693    Outdazzle Designer Scorpio for Men's Suit, Jac...
1019                 Autofurnish Car Cover For Santro Xing
1772     AMZER Back Cover for Samsung Galaxy Tab A 9.7 ...
10692                             Tangerine Crystal Brooch
Name: product_name, dtype: object

## Summary

This notebook builds a simple but functional content-based product recommender:
- Cleaned and combined text fields (name, brand, category, description) into a single representation per product
- Converted that text into TF-IDF vectors and computed cosine similarity between all products
- Wrapped it in a `product_recommendation()` function that returns the top 10 most similar products to a given item

**Observations / next steps:**
- Description-only similarity (`des_sim`) gave noticeably more coherent recommendations than the combined-metadata version (`overall_sim`) in the examples above — worth investigating why (possibly the category-tree text needs more cleanup, or fields need different weighting).
- Could also compare against a distance-based approach using `pairwise_distances` / `MultiLabelBinarizer` on category tags.
- No user interaction data was available here, so this is purely content-based — a natural extension would be a hybrid model if user ratings/click data becomes available.
